In [2]:
import pandas as pd

csv_file = r"E:\大学\学习\血液激素神经网络\archive\all_stocks_5yr.csv"
df = pd.read_csv(csv_file)
print(df.columns)
print(df.head())

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'Name'], dtype='str')
         date   open   high    low  close    volume Name
0  2013-02-08  15.07  15.12  14.63  14.75   8407500  AAL
1  2013-02-11  14.89  15.01  14.26  14.46   8882000  AAL
2  2013-02-12  14.45  14.51  14.10  14.27   8126000  AAL
3  2013-02-13  14.30  14.94  14.25  14.66  10259500  AAL
4  2013-02-14  14.94  14.96  13.16  13.99  31879900  AAL


In [4]:
import pandas as pd
import numpy as np
import torch
import os

# ================= 参数设置 =================
csv_file = r"E:\大学\学习\血液激素神经网络\archive\all_stocks_5yr.csv"
output_dir = r"E:\大学\学习\血液激素神经网络\archive\processed"
os.makedirs(output_dir, exist_ok=True)

tickers = ['AAPL','MSFT','GOOGL','AMZN','TSLA']
T = 30  # 用前30天数据预测下一天

# ================= RSI 计算函数 =================
def compute_RSI(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.rolling(period).mean()
    roll_down = down.rolling(period).mean()
    rs = roll_up / roll_down
    rsi = 100 - (100 / (1 + rs))
    return rsi

# ================= 技术指标函数 =================
def add_technical_indicators(df, ticker):
    close = df[f"{ticker}_Close"]
    vol = df[f"{ticker}_Volume"]

    # 收益率
    df[f"{ticker}_Return"] = close.pct_change()
    df[f"{ticker}_LogReturn"] = np.log(close) - np.log(close.shift(1))

    # 均线
    for n in [5,10,20,50]:
        df[f"{ticker}_SMA_{n}"] = close.rolling(n).mean()
        df[f"{ticker}_EMA_{n}"] = close.ewm(span=n, adjust=False).mean()
        df[f"{ticker}_Close_MA_diff_{n}"] = close - df[f"{ticker}_SMA_{n}"]

    # 波动率 & 布林带
    for n in [10,20]:
        df[f"{ticker}_Volatility_{n}"] = close.rolling(n).std()
        df[f"{ticker}_BB_Upper_{n}"] = df[f"{ticker}_SMA_{n}"] + 2*df[f"{ticker}_Volatility_{n}"]
        df[f"{ticker}_BB_Lower_{n}"] = df[f"{ticker}_SMA_{n}"] - 2*df[f"{ticker}_Volatility_{n}"]

    # 动量
    df[f"{ticker}_RSI_6"] = compute_RSI(close,6)
    df[f"{ticker}_RSI_14"] = compute_RSI(close,14)
    df[f"{ticker}_Momentum_10"] = close - close.shift(10)

    # 成交量
    df[f"{ticker}_Volume_MA5"] = vol.rolling(5).mean()
    df[f"{ticker}_Volume_MA10"] = vol.rolling(10).mean()
    df[f"{ticker}_Volume_Change"] = vol.pct_change()

    # MACD
    exp1 = close.ewm(span=12, adjust=False).mean()
    exp2 = close.ewm(span=26, adjust=False).mean()
    df[f"{ticker}_MACD"] = exp1 - exp2
    df[f"{ticker}_MACD_Signal"] = df[f"{ticker}_MACD"].ewm(span=9, adjust=False).mean()

    return df

# ================= 读取 CSV =================
df = pd.read_csv(csv_file)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

# ================= 按股票处理并合并 =================
processed_df = pd.DataFrame(index=df.index.unique())

for t in tickers:
    df_t = df[df['Name']==t].copy()
    df_t = df_t.sort_index()

    # 重命名列
    df_t.rename(columns={
        'open': f'{t}_Open',
        'high': f'{t}_High',
        'low': f'{t}_Low',
        'close': f'{t}_Close',
        'volume': f'{t}_Volume'
    }, inplace=True)

    # 保留 Close 和 Volume 列
    df_t = df_t[[f'{t}_Open', f'{t}_High', f'{t}_Low', f'{t}_Close', f'{t}_Volume']]

    # 添加技术指标
    df_t = add_technical_indicators(df_t, t)

    # 合并到总表
    processed_df = processed_df.join(df_t, how='outer')

# 填充缺失值
# 填充缺失值（兼容旧版本 pandas）
processed_df = processed_df.ffill()
processed_df = processed_df.bfill()

# 保存处理后的 CSV
processed_csv = os.path.join(output_dir, "all_stocks_5yr_processed.csv")
processed_df.to_csv(processed_csv)
print("💾 技术指标添加完成并保存:", processed_csv)

# ================= 转换成模型输入 (X, y) =================
feature_cols = []
for t in tickers:
    cols = [c for c in processed_df.columns if c.startswith(t) and c not in ['AdjClose']]
    feature_cols += cols

features = processed_df[feature_cols].values
X, y = [], []

for i in range(len(features)-T):
    X.append(features[i:i+T])
    y.append(features[i+T, [feature_cols.index(f"{t}_Close") for t in tickers]])

X = torch.tensor(np.array(X), dtype=torch.float32)
y = torch.tensor(np.array(y), dtype=torch.float32)

print("✅ X shape:", X.shape)
print("✅ y shape:", y.shape)

💾 技术指标添加完成并保存: E:\大学\学习\血液激素神经网络\archive\processed\all_stocks_5yr_processed.csv
✅ X shape: torch.Size([1229, 30, 165])
✅ y shape: torch.Size([1229, 5])
